# Extract Routing Trace from TinyMoE-100m-2x8

**Corrected version v2** (single forward pass, no duplicates)

Changes:
- Removed model.generate() - was causing N forward passes with growing context
- Single model(**inputs) forward pass - clean trace without duplicates
- max_length=1000 (model limit is 1024)
- Removed unused _current_layer
- Added Trace size validation

Expected output:
```
Input sequence length: ~1000
Trace size: 10000 records
Expected: 1000 tokens x 10 layers = 10000
```

In [ ]:
# Install Rust 1.93.0
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.93.0
!rustup default 1.93.0
!rustc --version

In [ ]:
# Install dependencies for trace extraction
!pip install torch transformers accelerate

In [ ]:
# ============================================================
# extract_routing_trace.py — Colab-ready (v2, single forward)
# ============================================================
# Извлекает routing trace из FlameF0X/TinyMoE-100m-2x8.
#
# Выход: /content/routing-trace.jsonl
# Формат: одна строка на (layer, token_pos):
#   {"layer": 0, "pos": 0, "experts": [3, 7], "weights": [0.62, 0.38]}
#
# Почему один forward pass, а не generate():
#   generate() вызывает forward на каждом шаге с растущим контекстом.
#   Это создаёт дубли: N*(N+1)/2 записей вместо N.
#   Один forward pass даёт чистый trace: N токенов × M слоёв.
# ============================================================

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

MODEL_ID = "FlameF0X/TinyMoE-100m-2x8"
OUT_PATH = "/content/routing-trace.jsonl"

# ------------------------------------------------------------
# 1. Загрузка модели и токенизатора
# ------------------------------------------------------------
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto",
)
model.eval()

config = model.config
print(f"num_hidden_layers = {config.num_hidden_layers}")
print(f"num_experts = {getattr(config, 'num_local_experts', getattr(config, 'num_experts', '?'))}")
print(f"num_experts_per_tok = {getattr(config, 'num_experts_per_tok', '?')}")

# ------------------------------------------------------------
# 2. Хранилище трассировки
# ------------------------------------------------------------
trace = []

# ------------------------------------------------------------
# 3. Hook — исправленная версия
# ------------------------------------------------------------
def make_hook(layer_idx):
    def hook(module, args, output):
        if isinstance(output, tuple) and len(output) >= 3:
            _, top_k_weights, top_k_index = output[0], output[1], output[2]
        elif isinstance(output, tuple) and len(output) == 2:
            top_k_index = output[1]
            top_k_weights = None
        else:
            router_logits = output if not isinstance(output, tuple) else output[0]
            top_k = getattr(module, "top_k", 2)
            top_k_weights, top_k_index = torch.topk(
                torch.softmax(router_logits, dim=-1), top_k, dim=-1
            )

        if top_k_index.dim() == 3:
            top_k_index = top_k_index.reshape(-1, top_k_index.shape[-1])
        if top_k_weights is not None and top_k_weights.dim() == 3:
            top_k_weights = top_k_weights.reshape(-1, top_k_weights.shape[-1])

        n_tokens = top_k_index.shape[0]

        if layer_idx == 0:
            print(f"[hook] layer={layer_idx} index_shape={tuple(top_k_index.shape)}")

        for pos in range(n_tokens):
            exp_list = sorted(top_k_index[pos].tolist())
            w_list = (
                top_k_weights[pos].tolist()
                if top_k_weights is not None
                else [None] * len(exp_list)
            )
            trace.append({
                "layer": layer_idx,
                "pos": pos,
                "experts": exp_list,
                "weights": w_list,
            })

    return hook

# ------------------------------------------------------------
# 4. Регистрация хуков
# ------------------------------------------------------------
hooks = []
for name, module in model.named_modules():
    if name.endswith(".mlp.gate"):
        parts = name.split(".")
        layer_idx = None
        for i, p in enumerate(parts):
            if p == "layers" and i + 1 < len(parts):
                try:
                    layer_idx = int(parts[i + 1])
                except ValueError:
                    pass
                break
        if layer_idx is None:
            continue
        h = module.register_forward_hook(make_hook(layer_idx))
        hooks.append(h)
        print(f"Hooked layer {layer_idx}: {name}")

print(f"Total hooks: {len(hooks)}")

# ------------------------------------------------------------
# 5. Один forward pass, без generate
# ------------------------------------------------------------
# Модель имеет max context 1024. Берём ~900 токенов, один проход.
# Никаких generate — это устраняет дубли в trace.

# Разные тексты, чтобы routing не был однообразным от одного шаблона
seed_text = (
    "The quick brown fox jumps over the lazy dog. "
    "Machine learning models process text token by token. "
    "Mixture of experts routes each token to specialized subnetworks. "
    "Storage layouts affect memory access patterns. "
    "Cold reads hit the disk, warm reads hit the cache. "
) * 20

inputs = tokenizer(
    seed_text,
    return_tensors="pt",
    truncation=True,
    max_length=1000,   # запас от 1024
).to(model.device)

n_input = inputs["input_ids"].shape[1]
print(f"Input sequence length: {n_input}")

print("Running single forward pass...")
with torch.no_grad():
    _ = model(**inputs)

# ------------------------------------------------------------
# 6. Удаление хуков
# ------------------------------------------------------------
for h in hooks:
    h.remove()
print("Hooks removed.")
print(f"Trace size: {len(trace)} records")
print(f"Expected: {n_input} tokens x {len(hooks)} layers = {n_input * len(hooks)}")

# ------------------------------------------------------------
# 7. Запись JSONL
# ------------------------------------------------------------
trace.sort(key=lambda r: (r["layer"], r["pos"]))

with open(OUT_PATH, "w") as f:
    for rec in trace:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(trace)} records to {OUT_PATH}")

# ------------------------------------------------------------
# 8. Быстрая сводка — распределение экспертов
# ------------------------------------------------------------
from collections import Counter

per_layer = {}
for rec in trace:
    per_layer.setdefault(rec["layer"], Counter()).update(rec["experts"])

print("\nExpert frequency per layer (top 3):")
for layer in sorted(per_layer):
    top = per_layer[layer].most_common(3)
    print(f"  layer {layer}: {top}")

# ------------------------------------------------------------
# 9. Скачивание файла
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(OUT_PATH)
except ImportError:
    print(f"(not in Colab; file at {OUT_PATH})")

In [ ]:
# Optional: build forge and run bench-read
!cargo build --release -p forge-cli
# Upload manifest.json from your machine, then:
# !./target/release/forge bench-read --manifest manifest.json --trace routing-trace.jsonl --direct --out bench-real.jsonl